In [34]:
# Test Neo4j connection
from neo4j import GraphDatabase

uri = "bolt://localhost:7687"
user = "neo4j"
pwd = "your_local_password"

try:
    driver = GraphDatabase.driver(uri, auth=(user, pwd))
    with driver.session() as session:
        result = session.run("RETURN 'Hello, Neo4j!' AS message")
        record = result.single()
        print(record["message"])
    driver.close()
    print("Connection successful!")
except Exception as e:
    print(f"Connection failed: {e}")

Unable to retrieve routing information


Connection failed: Unable to retrieve routing information


In [25]:
# Query the knowledge graph
from neo4j import GraphDatabase
import os
import dotenv

dotenv.load_dotenv()

uri = "bolt://localhost:7687"
user = "neo4j"
pwd = os.getenv("password")

driver = GraphDatabase.driver(uri, auth=(user, pwd))

with driver.session() as session:
    # Count nodes
    result = session.run("MATCH (n) RETURN labels(n) as labels, count(*) as count")
    print("Node counts:")
    for record in result:
        print(f"  {record['labels']}: {record['count']}")
    
    # Check for descriptions
    result = session.run("MATCH (a:ArtPiece) WHERE a.description IS NOT NULL RETURN a.title as title, a.description as description LIMIT 5")
    print("\nArtworks with descriptions:")
    for record in result:
        print(f"  {record['title']}: {record['description'][:100]}...")

driver.close()

Node counts:
  ['Palau']: 3
  ['Sala']: 6
  ['ArtPiece']: 72
  ['Artist']: 23
  ['Theme']: 38

Artworks with descriptions:
  Sant Jeroni penitent: (Entre les taules de Sant Jeroni i els Ressuscitats) Aquestes dues pintures sobre taula ens ajuden a...
  Davallament: Moltes de les obres d’art que avui veiem dins d’un museu han perdut el context i la funció originals...
  Mare de Déu de la Llet: Moltes de les obres d’art que avui veiem dins d’un museu han perdut el context i la funció originals...
  Flagel·lació de Crist: Dins del conjunt destaca una escultura en fusta policromada. La imatge de la Mare de Déu de Trapani ...


In [ ]:
# Add room descriptions to the knowledge graph
from docx import Document
from neo4j import GraphDatabase

def read_docx(file_path):
    doc = Document(file_path)
    text = ""
    for para in doc.paragraphs:
        if para.text.strip():
            text += para.text + "\n"
    return text

def add_room_descriptions(uri, user, pwd):
    room_texts = read_docx("/GuIA/raw_data/Textos_de_sala_MUSEU_DEL_RENAIXEMENT.docx")

    driver = GraphDatabase.driver(uri, auth=(user, pwd))

    # Parse room texts - they seem to be labeled as "Text X – PB Y" or similar
    sections = room_texts.split("Text ")
    for section in sections[1:]:  # Skip the header
        lines = section.split('\n')
        if lines:
            header = lines[0].strip()
            content = '\n'.join(lines[1:]).strip()
            
            # Try to extract room info, e.g., "1 – PB 0" or "2 – PB 0"
            if ' – ' in header:
                text_num, location = header.split(' – ', 1)
                if 'PB' in location or 'P' in location:
                    # Map PB 0 to palau 1, sala 0 or something
                    # Assuming PB means Planta Baixa (Ground Floor), perhaps palau 1
                    if 'PB 0' in location:
                        palau_id = '1'
                        sala_id = '0'
                    elif 'PB 1' in location:
                        palau_id = '1'
                        sala_id = '1'
                    else:
                        continue
                    
                    with driver.session() as session:
                        session.run("""
                        MATCH (s:Sala {id: $sala, palau: $palau})
                        SET s.description = $description
                        """, sala=sala_id, palau=palau_id, description=content)
                    print(f"Added description to Palau {palau_id}, Sala {sala_id}")

    driver.close()
    print("Room descriptions added!")

# Run the function
add_room_descriptions("bolt://localhost:7687", "neo4j", pwd)

Added description to Palau 1, Sala 0
Added description to Palau 1, Sala 0
Added description to Palau 1, Sala 1
Room descriptions added!


In [ ]:
# Improved loading with Artist nodes
import pandas as pd
from neo4j import GraphDatabase

def parse_ubic(ubic_str):
    if not isinstance(ubic_str, str) or '-' not in ubic_str:
        return "Unknown", "Unknown"
    parts = ubic_str.split('-')
    palau = parts[0].replace('P', '')
    sala = parts[1].replace('S', '')
    return palau, sala

def load_museum_excel_improved(file_path, uri, user, pwd):
    df = pd.read_excel(file_path, header=None)
    df = df[1:]  # skip title row
    df.columns = df.iloc[0]  # set headers
    df = df[1:]  # skip header row
    
    driver = GraphDatabase.driver(uri, auth=(user, pwd))
    
    with driver.session() as session:
        for _, row in df.iterrows():
            palau_id, sala_id = parse_ubic(row['UBIC.'])
            artist_name = row['AUTORIA']
            
            # Create ArtPiece
            session.run("""
            MERGE (p:Palau {id: $palau})
            MERGE (s:Sala {id: $sala, palau: $palau})
            MERGE (p)-[:HAS_SALA]->(s)
            
            MERGE (a:ArtPiece {title: $title})
            SET a.artist = $artist, 
                a.dating = $dating,
                a.technique = $technique
            MERGE (a)-[:LOCATED_IN]->(s)
            """, 
            palau=palau_id, sala=sala_id, 
            title=row['TÍTOL / DESCRIPCIÓ'], 
            artist=artist_name,
            dating=row['DATACIÓ'],
            technique=row['TÈCNICA / TIPOLOGIA'])
            
            # Link to artist if not anonymous
            if pd.notna(artist_name) and 'Anònim' not in artist_name:
                session.run("""
                MERGE (art:Artist {name: $name})
                MERGE (a:ArtPiece {title: $title})
                MERGE (a)-[:CREATED_BY]->(art)
                """, name=artist_name, title=row['TÍTOL / DESCRIPCIÓ'])
    
    # Create relationships between artists
    with driver.session() as session:
        session.run("""
        MATCH (a1:Artist)-[:CREATED_BY]-(art1:ArtPiece), (a2:Artist)-[:CREATED_BY]-(art2:ArtPiece)
        WHERE a1 <> a2 AND art1.dating = art2.dating AND art1.dating IS NOT NULL
        MERGE (a1)-[:CONTEMPORARY_OF {period: art1.dating}]->(a2)
        """)
    
    driver.close()
    print("Improved data loaded with Artist nodes and relationships!")

# Run the improved loading
load_museum_excel_improved("/Users/neromelt/UAB/Synthesys II/GuIA/raw_data/2026_obres_Museu_del_Renaixement.xlsx", 
                           "bolt://localhost:7687", "neo4j", pwd)

Improved data loaded with Artist nodes and relationships!


In [ ]:
# Create Theme nodes from audio guide
from docx import Document
from neo4j import GraphDatabase

def read_docx(file_path):
    doc = Document(file_path)
    text = ""
    for para in doc.paragraphs:
        if para.text.strip():
            text += para.text + "\n"
    return text

def create_themes_from_audio_guide(uri, user, pwd):
    audio_text = read_docx("/GuIA/raw_data/Guió_Audioguia_Museu_Renaixement.docx")
    
    driver = GraphDatabase.driver(uri, auth=(user, pwd))
    
    # Extract sections as themes
    lines = audio_text.split('\n')
    current_theme = None
    theme_content = ""
    
    for line in lines:
        line = line.strip()
        if line and not line.startswith(' ') and len(line) < 100:  # Likely a section title
            if current_theme:
                # Save previous theme
                with driver.session() as session:
                    session.run("""
                    MERGE (t:Theme {name: $name})
                    SET t.description = $description
                    """, name=current_theme, description=theme_content.strip())
            
            current_theme = line
            theme_content = ""
        else:
            theme_content += line + " "
    
    # Save last theme
    if current_theme:
        with driver.session() as session:
            session.run("""
            MERGE (t:Theme {name: $name})
            SET t.description = $description
            """, name=current_theme, description=theme_content.strip())
    
    # Link themes to rooms based on content (simple approach)
    with driver.session() as session:
        # For example, link "Primera planta" themes to palau 1
        session.run("""
        MATCH (t:Theme) WHERE t.name CONTAINS 'Primera'
        MATCH (p:Palau {id: '1'})
        MERGE (p)-[:FEATURES_THEME]->(t)
        """)
        
        session.run("""
        MATCH (t:Theme) WHERE t.name CONTAINS 'Planta baixa'
        MATCH (p:Palau {id: '1'})
        MERGE (p)-[:FEATURES_THEME]->(t)
        """)
    
    driver.close()
    print("Themes created from audio guide!")

# Run theme creation
create_themes_from_audio_guide("bolt://localhost:7687", "neo4j", pwd)

Themes created from audio guide!
